# 🌍 Multilingual Dubber MVP — Qwen3-TTS

This notebook demonstrates Qwen3-TTS's capabilities across all 10 supported languages. It will generate audio in each language and also show cross-lingual cloning (designing a voice in English and speaking other languages with it).

### Supported Languages
| Language | Native Name |
| --- | --- |
| English | English |
| Chinese | 中文 (Zhōngwén) |
| Japanese | 日本語 (Nihongo) |
| Korean | 한국어 (Hangugeo) |
| German | Deutsch |
| French | Français |
| Spanish | Español |
| Italian | Italiano |
| Russian | Русский |
| Portuguese | Português |


In [ ]:
!pip install -q qwen-tts soundfile
import os
import gc
import torch
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

os.makedirs('outputs', exist_ok=True)


In [ ]:
print("Loading CustomVoice Model...")
cv_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

LANGUAGES = {
    "English": "Hello, welcome to the future of voice AI! This is Qwen3-TTS.",
    "Chinese": "你好，欢迎来到语音人工智能的未来！这是通义语音。",
    "Japanese": "こんにちは、音声AIの未来へようこそ！これはQwen3-TTSです。",
    "Korean": "안녕하세요, 음성 AI의 미래에 오신 것을 환영합니다! 이것은 Qwen3-TTS입니다.",
    "German": "Hallo, willkommen in der Zukunft der Sprach-KI! Das ist Qwen3-TTS.",
    "French": "Bonjour, bienvenue dans le futur de l'IA vocale ! Voici Qwen3-TTS.",
    "Spanish": "Hola, bienvenido al futuro de la inteligencia artificial de voz. Esto es Qwen3-TTS.",
    "Italian": "Ciao, benvenuto nel futuro dell'intelligenza artificiale vocale! Questo è Qwen3-TTS.",
    "Russian": "Привет, добро пожаловать в будущее голосового ИИ! Это Qwen3-TTS.",
    "Portuguese": "Olá, bem-vindo ao futuro da IA de voz! Este é o Qwen3-TTS."
}


In [ ]:
print("Generating audio in all supported languages...")
for lang, text in LANGUAGES.items():
    print(f"\nGenerating {lang}...")
    audio = cv_model.generate_custom_voice(text=text, language=lang.lower(), speaker="Ryan", instruct="")
    
    out_path = f"outputs/lang_{lang.lower()}.wav"
    sf.write(out_path, audio, samplerate=24000)
    
    print(f"{lang}: {text}")
    display(Audio(out_path))


In [ ]:
print("Clearing VRAM for cross-lingual cloning...")
del cv_model
gc.collect()
torch.cuda.empty_cache()

print("Loading VoiceDesign Model...")
vd_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

eng_text = "This is my custom designed voice, and soon I'll be speaking other languages."
eng_audio = vd_model.generate_voice_design(
    text=eng_text,
    language="english",
    instruct="A deep, resonant male voice with a calm and wise tone"
)
sf.write("outputs/custom_english_ref.wav", eng_audio, samplerate=24000)
print("Custom English Voice Design Reference:")
display(Audio("outputs/custom_english_ref.wav"))

print("Clearing VRAM again...")
del vd_model
gc.collect()
torch.cuda.empty_cache()

print("Loading Base Model for cloning...")
base_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

print("Creating voice clone prompt from the English reference...")
clone_prompt = base_model.create_voice_clone_prompt("outputs/custom_english_ref.wav", eng_text)

cross_lingual_tests = {
    "chinese": "现在我能用中文说话了。这真是太神奇了。",
    "french": "Maintenant, je peux parler français. C'est vraiment incroyable.",
    "japanese": "これで私は日本語を話すことができます。本当に素晴らしいです。"
}

for lang, text in cross_lingual_tests.items():
    print(f"\nCloning into {lang.capitalize()}...")
    cloned_audio = base_model.generate_voice_clone(
        text=text,
        language=lang,
        voice_clone_prompt=clone_prompt
    )
    path = f"outputs/cloned_{lang}.wav"
    sf.write(path, cloned_audio, samplerate=24000)
    display(Audio(path))


In [ ]:
import pandas as pd
import os

stats = []
for file in os.listdir("outputs"):
    if file.endswith(".wav"):
        path = os.path.join("outputs", file)
        size_kb = os.path.getsize(path) / 1024
        info = sf.info(path)
        stats.append({
            "File": file,
            "Duration (s)": round(info.duration, 2),
            "Size (KB)": round(size_kb, 1)
        })

df = pd.DataFrame(stats)
display(df)


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('multilingual_outputs', 'zip', 'outputs')
files.download('multilingual_outputs.zip')
